# Backpropagation From Scratch

**Topic:** MLPs and Backpropagation &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(1600, 2)`, `y_train` `(1600,)`: the classic two interleaving half-moons with noise (labels 0/1).
- `X_test` `(500, 2)`: predict these. The evaluation cell rebuilds the true labels from the same seed.

## Your task

With **NumPy only** (no sklearn models, no autograd):

1. Build an MLP with one hidden layer (~16 tanh or ReLU units) and a sigmoid output.
2. Derive and implement the backward pass for binary cross-entropy (hint: the output-layer error simplifies to `p − y`).
3. Train with gradient descent, then set `predictions` to the 0/1 labels your trained model predicts for `X_test` (a length-500 array of 0s and 1s).

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
from sklearn.datasets import make_moons

X_train, y_train = make_moons(n_samples=1600, noise=0.22, random_state=221)
X_test, _ = make_moons(n_samples=500, noise=0.22, random_state=222)
print(f"X_train {X_train.shape}. Plot it to see the moons")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np
from sklearn.datasets import make_moons

TARGET_ACCURACY = 0.88

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

# The true test labels are rebuilt here from the same seed as X_test,
# so they never sit in the solution cell's namespace.
_, _true = make_moons(n_samples=500, noise=0.22, random_state=222)
_pred = np.asarray(predictions).ravel()
_true = np.asarray(_true).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {int(_cls):>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

One hidden tanh layer with a sigmoid output. The backward pass uses the BCE+sigmoid shortcut `dL/dz_out = (p - y)/n`, then pushes that error through `w2` and the `1 - tanh^2` derivative, the same four equations from the lesson.

```python
import numpy as np

rng = np.random.default_rng(0)
hidden = 16
W1 = rng.normal(0, 1.0, (2, hidden))
b1 = np.zeros(hidden)
w2 = rng.normal(0, 1.0, hidden)
b2 = 0.0

X, y = X_train, y_train.astype(float)
lr = 1.0
for _ in range(4000):
    # forward
    H = np.tanh(X @ W1 + b1)
    p = 1.0 / (1.0 + np.exp(-(H @ w2 + b2)))
    # backward (BCE + sigmoid shortcut: dL/dz = p - y)
    dz = (p - y) / len(y)
    gw2 = H.T @ dz
    gb2 = dz.sum()
    dH = np.outer(dz, w2)
    dZ = dH * (1.0 - H ** 2)
    gW1 = X.T @ dZ
    gb1 = dZ.sum(axis=0)
    W1 -= lr * gW1; b1 -= lr * gb1
    w2 -= lr * gw2; b2 -= lr * gb2

H = np.tanh(X_test @ W1 + b1)
p = 1.0 / (1.0 + np.exp(-(H @ w2 + b2)))
predictions = (p >= 0.5).astype(int)
```
